# Non-Markovianity estimations (Lorentzian-spectrum dataset)
## (Breuer measure approximation, ported from `NonMarkovianity.ipynb`)

In [ ]:
using LinearAlgebra
using Combinatorics
using HDF5

include("LiPoSID.jl")

Trace distance:

$D(\rho_1, \rho_2) = \frac{1}{2} \operatorname{Tr}|\rho_1 - \rho_2|$

Non-Markovianity (H.-P. Breuer, E.-M. Laine, J. Piilo, PRL 2009):

$\sigma = \frac{dD}{dt}, \quad \mathcal{N} = \max_{\rho_1(0), \rho_2(0)}{\int_{\sigma>0}{\sigma dt}}$

In [ ]:
function NonMarkovianityLorentzian(data_dir, states, widᵢ)
    N = []
    for (i,j) in combinations(1:length(states), 2)
        t₁, ρs₁ = LiPoSID.read_lorentzian_timeevolution(data_dir, states[i], widᵢ)
        t₂, ρs₂ = LiPoSID.read_lorentzian_timeevolution(data_dir, states[j], widᵢ)
        dD = diff([LiPoSID.TrDist(ρ₁, ρ₂) for (ρ₁, ρ₂) in zip(ρs₁, ρs₂)])
        push!(N, sum(dD[dD .> 0]))
    end
    maximum(N)
end

In [ ]:
data_dir = "DATA/"

dodeca_states = ["Dodeca"*string(n) for n=1:10]
basis_states  = ["State0", "State1", "StateX", "StateY"]
all_states    = vcat(basis_states, dodeca_states)

widths = ["wid1", "wid2", "wid3", "wid4", "wid5"]
width_value = Dict("wid1"=>64.0, "wid2"=>16.0, "wid3"=>4.0, "wid4"=>2.0, "wid5"=>1.0)

In [ ]:
# sanity check on one width before running the full loop
NonMarkovianityLorentzian(data_dir, all_states[1:3], widths[end])

In [ ]:
N = [NonMarkovianityLorentzian(data_dir, all_states, widᵢ) for widᵢ in widths]

In [ ]:
# Save so 03C_plot_violin_tracedist_Lorentzian.ipynb (Python) can load it
using PyCall
pickle = pyimport("pickle"); pyopen = pyimport("builtins").open
file = pyopen("NonMark_Lorentzian.pkl", "wb"); pickle.dump(N, file); file.close()
println("Saved NonMark_Lorentzian.pkl : ", N)